# Synergy prediction using transcriptional interaction scores

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
%matplotlib inline
from pathlib import Path

# Configure root
root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")

else:
  l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge"
  cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/cfus"

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Bliss score and simple interaction score
data_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
data_df = data_df.dropna(axis = 1)

## Training with stratified split

Nested CV for ElasticNet.

In [ ]:
from src.split import combination_stratified_split
from src.train import run_nested_elasticnet_cv

# Make stratified splits
strat_splits = combination_stratified_split(data_df, num_folds = 5, seed = 111)

# Run Nested CV
scores1, mean_score1 = run_nested_elasticnet_cv(
    df = data_df,
    splits = strat_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores1}")
print(f"Mean R^2 : {mean_score1}")

Nested CV for PLS regression.

In [ ]:
from src.train import run_nested_pls_cv

scores2, mean_score2 = run_nested_pls_cv(
    df = data_df,
    splits = strat_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores2}")
print(f"Mean R^2 : {mean_score2}")

## Training with held-out drug combination

Nested CV for ElasticNet.

In [ ]:
from src.split import combination_held_out_split

# Make held out splits
held_out_splits = combination_held_out_split(data_df)

scores3, mean_score3 = run_nested_elasticnet_cv(
    df = data_df,
    splits = held_out_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores3}")
print(f"Mean R^2 : {mean_score3}")

Nested CV for PLS regression.

In [ ]:
scores4, mean_score4 = run_nested_pls_cv(
    df = data_df,
    splits = held_out_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores4}")
print(f"Mean R^2 : {mean_score4}")

## Training with held-out timepoint

NestedCV for ElasticNet.

In [ ]:
from src.split import timepoint_held_out_split

# Make time splits
time_splits = timepoint_held_out_split(data_df)

scores5, mean_score5 = run_nested_elasticnet_cv(
    df = data_df,
    splits = time_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores5}")
print(f"Mean R^2 : {mean_score5}")

NestedCV for PLS regression.

In [ ]:
time_splits = timepoint_held_out_split(data_df)

scores6, mean_score6 = run_nested_pls_cv(
    df = data_df,
    splits = time_splits,
    synergy = True
)

print(f"R^2 for 5 folds : {scores6}")
print(f"Mean R^2 : {mean_score6}")

## Training with sparse combination matrix.

Train on 2 combinations + sparse matrix of last combination.

In [ ]:
from src.train import train_custom_synergy_model

# CEF+CIP mask
cefcip_mask = (data_df["drug_id"] != "CEF+CIP") | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_synergy_model(
    df = data_df,
    train_mask = cefcip_mask,
    test_mask = ~cefcip_mask,
    title = "Results for PLS regression model with sparse CEF+CIP data held-out"
)

In [ ]:
# CIP+VNC mask
cipvnc_mask = (data_df["drug_id"] != "CIP+VNC") | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_synergy_model(
    df = data_df,
    train_mask = cipvnc_mask,
    test_mask = ~cipvnc_mask,
    title = "Results for PLS regression model with sparse CIP+VNC data held-out"
)

In [ ]:
# CEF+RIF mask
cefrif_mask = (data_df["drug_id"] != "CEF+RIF") | ((data_df["drug1_dose"]) == (data_df["drug2_dose"]))

train_custom_synergy_model(
    df = data_df,
    train_mask = cefrif_mask,
    test_mask = ~cefrif_mask,
    title = "Results for PLS regression model with sparse CEF+RIF data held-out"
)